# Data Audit

In [1]:
import pandas as pd
import numpy as np

 ## Load Datasets

In [2]:
# Load all CSV files from the raw data directory into a dictionary of DataFrames
from pathlib import Path

data_path = Path("../data/raw")
csv_files = list(data_path.glob("*.csv"))

datasets = {}
for file in csv_files:
    datasets[file.stem] = pd.read_csv(file)

## Dataset Shape & Overview

In [3]:
# Summarize each dataset: shape, duplicates, missing values, memory usage, and dtypes
def audit_datasets(datasets):
    rows = []
    for name, df in datasets.items():
        n_rows, n_cols = df.shape
        n_dup_rows = df.duplicated().sum()
        n_unique_rows = n_rows - n_dup_rows
        total_cells = n_rows * n_cols
        missing_cells = df.isna().sum().sum()
        
        rows.append({
            'file': name,
            'rows': n_rows,
            'cols': n_cols,
            'unique_rows': n_unique_rows,
            'dup_rows': n_dup_rows,
            'dup_%': round(n_dup_rows / n_rows * 100, 2) if n_rows else 0,
            'missing_cells': missing_cells,
            'missing_%': round(missing_cells / total_cells * 100, 2) if total_cells else 0,
            'memory_MB': round(df.memory_usage(deep=True).sum() / 1024**2, 2),
            'dtypes': df.dtypes.astype(str).value_counts().to_dict(),
        })
    
    return pd.DataFrame(rows).sort_values('rows', ascending=False).reset_index(drop=True)

# Run the audit and display results sorted by row count
audit_datasets(datasets)

,file,rows,cols,unique_rows,dup_rows,dup_%,missing_cells,missing_%,memory_MB,dtypes
0,olist_geolocation_dataset,1000163,5,738332,261831,26.18,0,0.00,146.09,"{'float64': 2, 'object': 2, 'int64': 1}"
1,olist_order_items_dataset,112650,7,112650,0,0.00,0,0.00,39.43,"{'object': 4, 'float64': 2, 'int64': 1}"
2,olist_order_payments_dataset,103886,5,103886,0,0.00,0,0.00,17.81,"{'object': 2, 'int64': 2, 'float64': 1}"
3,olist_customers_dataset,99441,5,99441,0,0.00,0,0.00,29.62,"{'object': 4, 'int64': 1}"
4,olist_orders_dataset,99441,8,99441,0,0.00,4908,0.62,58.97,{'object': 8}
5,olist_order_reviews_dataset,99224,7,99224,0,0.00,145903,21.01,42.75,"{'object': 6, 'int64': 1}"
6,olist_products_dataset,32951,9,32951,0,0.00,2448,0.83,6.79,"{'float64': 7, 'object': 2}"
7,olist_sellers_dataset,3095,4,3095,0,0.00,0,0.00,0.66,"{'object': 3, 'int64': 1}"
8,product_category_name_translation,71,2,71,0,0.00,0,0.00,0.01,{'object': 2}


 ## Column-Level Audit

In [4]:
# Flatten all columns across datasets into a single table with dtype and missing-rate info
col_detail = pd.DataFrame([
    {'file': name, 'position': i, 'column': col, 'dtype': str(df[col].dtype), 'missing value': round(df[col].isnull().mean() * 100, 3)}
    for name, df in datasets.items()
    for i, col in enumerate(df.columns)
])

# Run the audit and display results sorted by row count
col_detail

,file,position,column,dtype,missing value
0,olist_geolocation_dataset,0,geolocation_zip_code_prefix,int64,0.000
1,olist_geolocation_dataset,1,geolocation_lat,float64,0.000
2,olist_geolocation_dataset,2,geolocation_lng,float64,0.000
3,olist_geolocation_dataset,3,geolocation_city,object,0.000
4,olist_geolocation_dataset,4,geolocation_state,object,0.000
5,olist_order_payments_dataset,0,order_id,object,0.000
6,olist_order_payments_dataset,1,payment_sequential,int64,0.000
7,olist_order_payments_dataset,2,payment_type,object,0.000
8,olist_order_payments_dataset,3,payment_installments,int64,0.000
9,olist_order_payments_dataset,4,payment_value,float64,0.000


 ## Candidate Keys

In [5]:
# Map each dataset to its ID-like columns for key candidacy testing
id_columns = {
    "olist_orders_dataset": ["order_id", "customer_id"],
    "olist_customers_dataset": ["customer_id", "customer_unique_id"],
    "olist_order_items_dataset": ["order_id", "order_item_id", "product_id", "seller_id"],
    "olist_order_reviews_dataset": ["review_id", "order_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "olist_geolocation_dataset": ["geolocation_zip_code_prefix"],
    "product_category_name_translation": ["product_category_name"],
}

# Check single-column uniqueness: flag columns that could serve as primary keys
def check_single_keys(datasets, id_columns):
    rows = []
    for name, df in datasets.items():
        for col in id_columns.get(name, []):
            if col in df.columns:
                rows.append({
                    'file': name,
                    'column': col,
                    'unique': df[col].nunique(),
                    'rows': len(df),
                    'duplicated': df[col].duplicated().sum(),
                    'candidate key': '🟢' if df[col].duplicated().sum() == 0 and df[col].nunique() ==len(df) else '🔴'
                })
    return pd.DataFrame(rows)

# Run single-column key analysis
single_key = check_single_keys(datasets, id_columns)
single_key

,file,column,unique,rows,duplicated,candidate key
0,olist_geolocation_dataset,geolocation_zip_code_prefix,19015,1000163,981148,🔴
1,olist_order_payments_dataset,order_id,99440,103886,4446,🔴
2,olist_order_payments_dataset,payment_sequential,29,103886,103857,🔴
3,olist_sellers_dataset,seller_id,3095,3095,0,🟢
4,olist_order_items_dataset,order_id,98666,112650,13984,🔴
5,olist_order_items_dataset,order_item_id,21,112650,112629,🔴
6,olist_order_items_dataset,product_id,32951,112650,79699,🔴
7,olist_order_items_dataset,seller_id,3095,112650,109555,🔴
8,olist_order_reviews_dataset,review_id,98410,99224,814,🔴
9,olist_order_reviews_dataset,order_id,98673,99224,551,🔴


In [6]:
# Define composite key candidates (multi-column combinations) for key testing
id_composite_columns = {
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_reviews_dataset": ["review_id", "order_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
}

# Check whether each column combination uniquely identifies rows (no duplicates, no nulls)
def analyze_duplicates(datasets, id_columns):
    rows = []
    for name, df in datasets.items():
        cols = id_columns.get(name, [])
        cols = [c for c in cols if c in df.columns]
        if not cols:
            continue
        rows.append({
            'file': name,
            'columns': ", ".join(cols),
            'unique': df[cols].drop_duplicates().shape[0],
            'rows': len(df),
            'duplicated': df.duplicated(subset=cols).sum(),
            'nulls': int(df[cols].isna().sum().sum()),
            'candidate key': '🟢' if (df.duplicated(subset=cols).sum() == 0 
                                       and df[cols].isna().sum().sum() == 0) 
                              else '🔴'
        })
    return pd.DataFrame(rows)

# Run composite key analysis
composite_key = analyze_duplicates(datasets, id_composite_columns)
composite_key

,file,columns,unique,rows,duplicated,nulls,candidate key
0,olist_order_payments_dataset,"order_id, payment_sequential",103886,103886,0,0,🟢
1,olist_order_items_dataset,"order_id, order_item_id",112650,112650,0,0,🟢
2,olist_order_reviews_dataset,"review_id, order_id",99224,99224,0,0,🟢


### Table Relationships Summary

| Table | Grain (one row represents...) | Rows | Candidate Key |
|---|---|---|---|
| orders | One order | 99,441 | order_id |
| customers | One customer record **per order** (not per person) | 99,441 | customer_id |
| order_items | One item within an order | 112,650 | order_id + order_item_id |
| payments | One payment record within an order | 103,886 | order_id + payment_sequential |
| reviews | One review record associated with an order | 99,224 | review_id + order_id |
| products | One product | 32,951 | product_id |
| sellers | One seller | 3,095 | seller_id |
| geolocation | One geolocation record (not unique per zip prefix) | 1,000,163 | — |
| translation | One product category and its English translation | 71 | product_category_name |


---
*customers → orders is 1:1 at the `customer_id` level because Olist generates a new `customer_id` for every order. The real customer is identified by `customer_unique_id`, which can map to multiple `customer_id`s (i.e. multiple orders by the same person). Repeat-customer analysis must use `customer_unique_id`, not `customer_id`.

*The orders → reviews relationship requires further investigation because order_id is not unique in the reviews table.

| Parent Table | Parent Key | Child Table | Foreign Key | Relationship |
|---|---|---|---|---|
| `customers` | `customer_id` | `orders` | `customer_id` | 1:N |
| `orders` | `order_id` | `order_items` | `order_id` | 1:N |
| `products` | `product_id` | `order_items` | `product_id` | 1:N |
| `sellers` | `seller_id` | `order_items` | `seller_id` | 1:N |
| `orders` | `order_id` | `payments` | `order_id` | 1:N |
| `orders` | `order_id` | `reviews` | `order_id` | 1:N* |
| `translation` | `product_category_name` | `products` | `product_category_name` | 1:N |

> *The `orders` → `reviews` relationship requires further investigation because `order_id` is not unique in the reviews table.

In [7]:
# Define relationships to validate: (child_table, child_col, parent_table, parent_col)
relationships = [
    ("olist_orders_dataset",         "customer_id", "olist_customers_dataset",  "customer_id"),
    ("olist_order_items_dataset",    "product_id",  "olist_products_dataset",   "product_id"),
    ("olist_order_items_dataset",    "seller_id",   "olist_sellers_dataset",    "seller_id"),
    ("olist_order_items_dataset",    "order_id",    "olist_orders_dataset",     "order_id"),
    ("olist_order_payments_dataset", "order_id",    "olist_orders_dataset",     "order_id"),
    ("olist_order_reviews_dataset",  "order_id",    "olist_orders_dataset",     "order_id"),
]

def validate_relationships(datasets, relationships):
    rows = []
    for child, child_col, parent, parent_col in relationships:
        if child not in datasets or parent not in datasets:
            continue
        c, p = datasets[child], datasets[parent]
        if child_col not in c.columns or parent_col not in p.columns:
            continue
        matched   = c[child_col].isin(p[parent_col]).sum()
        unmatched = len(c) - matched
        rows.append({
            "from": f"{child}.{child_col}",
            "to":   f"{parent}.{parent_col}",
            "matched": matched,
            "unmatched": unmatched,
            "match_%": round(matched / len(c) * 100, 2) if len(c) else 0,
            "valid": "🟢" if unmatched == 0 else "🔴",
        })
    return pd.DataFrame(rows)

validate_relationships(datasets, relationships)

,from,to,matched,unmatched,match_%,valid
0,olist_orders_dataset.customer_id,olist_customers_dataset.customer_id,99441,0,100.0,🟢
1,olist_order_items_dataset.product_id,olist_products_dataset.product_id,112650,0,100.0,🟢
2,olist_order_items_dataset.seller_id,olist_sellers_dataset.seller_id,112650,0,100.0,🟢
3,olist_order_items_dataset.order_id,olist_orders_dataset.order_id,112650,0,100.0,🟢
4,olist_order_payments_dataset.order_id,olist_orders_dataset.order_id,103886,0,100.0,🟢
5,olist_order_reviews_dataset.order_id,olist_orders_dataset.order_id,99224,0,100.0,🟢


### Coverage Gaps Between Orders and Child Tables

Checking the reverse direction (which orders have zero matching rows in each child table):

- `order_items`: 775 orders (0.78%) have no items
- `payments`: 1 order (0.00%) has no payment record
- `reviews`: 768 orders (0.77%) have no review

These gaps are expected — orders that were canceled or never paid for often never receive items, payment, or a review. This will be confirmed against `order_status` in the delivery/timing analysis below, but any join to `order_items`, `payments`, or `reviews` should be a **left join from orders**, not an inner join, or these orders will silently disappear from order-level metrics.

In [13]:
orders_df = datasets["olist_orders_dataset"]
order_ids = set(orders_df["order_id"])

for name in ["olist_order_items_dataset",
             "olist_order_payments_dataset",
             "olist_order_reviews_dataset"]:
    child_ids = set(datasets[name]["order_id"])
    missing = len(order_ids - child_ids)
    print(f"{name}: {missing} orders have no rows ({missing/len(order_ids):.2%})")

olist_order_items_dataset: 775 orders have no rows (0.78%)
olist_order_payments_dataset: 1 orders have no rows (0.00%)
olist_order_reviews_dataset: 768 orders have no rows (0.77%)


In [ ]:
reviews = datasets["olist_order_reviews_dataset"].copy()

duplicate_review_ids = (
    reviews[reviews["review_id"].duplicated(keep=False)]
    .sort_values("review_id")
)

duplicate_review_ids

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
31120,fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,NaN,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28 00:00:00,2018-02-28 13:57:52
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07


In [14]:
reviews["_msg"] = reviews["review_comment_message"].fillna("__NULL__")

review_group = reviews.groupby("order_id").agg(
    review_count=("review_id", "count"),
    score_count=("review_score", "nunique"),
    message_count=("_msg", "nunique"),
    has_null_message=("review_comment_message", lambda s: s.isna().sum()),
    has_text_message=("review_comment_message", lambda s: s.notna().sum()),
)

reviews = reviews.drop(columns="_msg")
review_group.query("review_count > 1").head(20)

,review_count,score_count,message_count,has_null_message,has_text_message
order_id,,,,,
0035246a40f520710769010f752e7507,2,1,2,1,1
013056cfe49763c6f66bda03396c5ee3,2,2,1,2,0
0176a6846bcb3b0d3aa3116a9a768597,2,1,1,2,0
02355020fd0a40a0d56df9f6ff060413,2,2,2,0,2
029863af4b968de1e5d6a82782e662f5,2,2,1,2,0
02e0b68852217f5715fb9cc885829454,2,1,2,0,2
02e723e8edb4a123d414f56cc9c4665e,2,1,2,1,1
03515a836bb855b03f7df9dee520a8fc,2,1,1,2,0
03c939fd7fd3b38f8485a0f95798f1f6,3,2,3,1,2


In [15]:
pattern_counts_v2 = (
    review_group
    .query("review_count > 1")
    .groupby(["review_count", "score_count", "message_count", 
              "has_null_message", "has_text_message"])
    .size()
    .reset_index(name="order_count")
    .sort_values("order_count", ascending=False)
)
pattern_counts_v2

,review_count,score_count,message_count,has_null_message,has_text_message,order_count
1,2,1,1,2,0,215
5,2,2,1,2,0,86
2,2,1,2,0,2,76
7,2,2,2,1,1,65
6,2,2,2,0,2,48
3,2,1,2,1,1,39
0,2,1,1,0,2,13
4,2,2,1,0,2,1
8,3,1,1,3,0,1
9,3,1,3,0,3,1


In [12]:
# Ensure date columns are datetime dtype
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

# Compute time spans between first and last review per order
time_diff = (
    reviews
    .groupby("order_id")
    .agg(
        n_reviews=("review_id", "count"),
        # Difference between latest and earliest creation date, in seconds
        creation_span=("review_creation_date", lambda s: (s.max() - s.min()).total_seconds()),
        # Difference between latest and earliest answer timestamp, in seconds
        answer_span=("review_answer_timestamp", lambda s: (s.max() - s.min()).total_seconds()),
    )
    .query("n_reviews > 1")  # Keep only orders with more than 1 review
)

time_diff.head()

,n_reviews,creation_span,answer_span
order_id,,,
0035246a40f520710769010f752e7507,2,345600.0,15195.0
013056cfe49763c6f66bda03396c5ee3,2,864000.0,881370.0
0176a6846bcb3b0d3aa3116a9a768597,2,0.0,41.0
02355020fd0a40a0d56df9f6ff060413,2,691200.0,697451.0
029863af4b968de1e5d6a82782e662f5,2,432000.0,252485.0


In [17]:
# Merge time-based features into review_group
review_group_t = review_group.join(
    time_diff[["creation_span", "answer_span"]],
    how="left"
)

# Convert seconds to hours for readability
review_group_t["creation_span_h"] = review_group_t["creation_span"] / 3600
review_group_t["answer_span_h"] = review_group_t["answer_span"] / 3600

# Show only multi-review orders
review_group_t.query("review_count > 1")[
    ["review_count", "score_count", "message_count",
     "has_null_message", "has_text_message",
     "creation_span_h", "answer_span_h"]
].head(10)

,review_count,score_count,message_count,has_null_message,has_text_message,creation_span_h,answer_span_h
order_id,,,,,,,
0035246a40f520710769010f752e7507,2,1,2,1,1,96.0,4.220833
013056cfe49763c6f66bda03396c5ee3,2,2,1,2,0,240.0,244.825000
0176a6846bcb3b0d3aa3116a9a768597,2,1,1,2,0,0.0,0.011389
02355020fd0a40a0d56df9f6ff060413,2,2,2,0,2,192.0,193.736389
029863af4b968de1e5d6a82782e662f5,2,2,1,2,0,120.0,70.134722
02e0b68852217f5715fb9cc885829454,2,1,2,0,2,48.0,0.015000
02e723e8edb4a123d414f56cc9c4665e,2,1,2,1,1,48.0,71.996667
03515a836bb855b03f7df9dee520a8fc,2,1,1,2,0,144.0,135.882500
03c939fd7fd3b38f8485a0f95798f1f6,3,2,3,1,2,552.0,556.643611


In [18]:
# Categorize orders by answer span
review_group_t["time_category"] = pd.cut(
    review_group_t["answer_span_h"],
    bins=[-1, 1/60, 1, 24, 30*24, float("inf")],
    labels=["<1min", "<1h", "<1d", "<30d", ">30d"]
)

# Only after the column exists, take the multi-review subset
multi_reviews = review_group_t.query("review_count > 1").copy()

pd.crosstab(
    [
        multi_reviews["review_count"],
        multi_reviews["score_count"],
        multi_reviews["has_null_message"],
        multi_reviews["has_text_message"]
    ],
    multi_reviews["time_category"]
)

time_category                                               <1min  <1h  <1d  \
review_count score_count has_null_message has_text_message                    
2            1           0                2                    11   19    8   
                         1                1                     3    5    3   
                         2                0                    58   26   23   
             2           0                2                     1    3    2   
                         1                1                     0    3    5   
                         2                0                    10    4    8   
3            1           0                3                     0    0    0   
                         3                0                     0    0    0   
             2           1                2                     0    0    0   
                         3                0                     0    1    0   

time_category                                               <30d  >30d  
review_count score_count has_null_message has_text_message              
2            1           0                2                   50     1  
                         1                1                   28     0  
                         2                0                  108     0  
             2           0                2                   41     2  
                         1                1                   54     3  
                         2                0                   60     4  
3            1           0                3                    1     0  
                         3                0                    1     0  
             2           1                2                    1     0  
                         3                0                    0     0

### Delivery Date Missingness by Order Status

`order_delivered_customer_date` is null for 2,957 orders overall, but this is almost entirely explained by `order_status`:

- Orders that are `canceled`, `unavailable`, `processing`, `invoiced`, `shipped`, `approved`, or `created` are expected to have no delivery date — the order was never completed.
- Only 8 orders marked `delivered` are missing a delivery date. These are a genuine data anomaly (8 out of 96,478 delivered orders, ~0.01%) and should be excluded or flagged separately rather than treated as "on time" or "late."

**Conclusion for delay analysis:** filter to `order_status == "delivered"` first, then treat any remaining null `order_delivered_customer_date` (8 rows) as a separate data-quality flag rather than a missing-at-random value.

In [19]:
# Pick one suspicious order and inspect its raw reviews
sample_order = review_group_t.query(
    "review_count > 1 and answer_span_h < 0.1"  # less than 6 minutes
).index[0]

reviews[reviews["order_id"] == sample_order][
    ["review_id", "review_score", "review_comment_message",
     "review_creation_date", "review_answer_timestamp"]
]

,review_id,review_score,review_comment_message,review_creation_date,review_answer_timestamp
854,830636803620cdf8b6ffaf1b2f6e92b2,5,NaN,2017-12-30,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,5,NaN,2017-12-30,2018-01-02 10:54:47


In [ ]:
# Check if duplicated review_ids have different timestamps
dup = reviews[reviews["review_id"].duplicated(keep=False)]
dup.groupby("review_id").agg(
    n_rows=("review_id", "size"),
    n_orders=("order_id", "nunique"),
    n_scores=("review_score", "nunique"),
    n_creation=("review_creation_date", "nunique"),
)

,n_rows,n_orders,n_scores,n_creation
review_id,,,,
00130cbe1f9d422698c812ed8ded1919,2,2,1,1
0115633a9c298b6a98bcbe4eee75345f,2,2,1,1
0174caf0ee5964646040cd94e15ac95e,2,2,1,1
017808d29fd1f942d97e50184dfb4c13,2,2,1,1
0254bd905dc677a6078990aad3331a36,2,2,1,1
...,...,...,...,...
fde2e6abaf5bb64f7407a44741c24dec,2,2,1,1
fde5986d35c89aa1b6ce4149de82a0d3,2,2,1,1
fe5c833752953fed3209646f1f63b53c,2,2,1,1


In [ ]:
# Check if duplicated review_ids have same answer timestamps
dup_check = (
    reviews[reviews["review_id"].duplicated(keep=False)]
    .groupby("review_id")
    .agg(
        n_rows=("review_id", "size"),
        n_orders=("order_id", "nunique"),
        n_answer=("review_answer_timestamp", "nunique"),
    )
)
dup_check["n_answer"].value_counts()

n_answer
1    789
Name: count, dtype: int64

In [22]:
dup = reviews[reviews["review_id"].duplicated(keep=False)].copy()
dup["_msg"] = dup["review_comment_message"].fillna("__NULL__")

g = dup.groupby("review_id").agg(
    n_rows=("review_id", "size"),
    n_orders=("order_id", "nunique"),
    n_scores=("review_score", "nunique"),
    n_msgs=("_msg", "nunique"),
    n_created=("review_creation_date", "nunique"),
    n_answer=("review_answer_timestamp", "nunique"),
)

print(g["n_rows"].value_counts())
print(g["n_orders"].value_counts())
for c in ["n_scores", "n_msgs", "n_created", "n_answer"]:
    print(c, (g[c] == 1).all())

# customer_id differs across the linked orders?
cust = (dup.merge(orders_df[["order_id", "customer_id"]], on="order_id")
           .groupby("review_id")["customer_id"].nunique())
print("customer differs in all groups:", (cust == g["n_orders"]).all())

n_rows
2    764
3     25
Name: count, dtype: int64
n_orders
2    764
3     25
Name: count, dtype: int64
n_scores True
n_msgs True
n_created True
n_answer True
customer differs in all groups: True


In [ ]:
# Load orders to check customer linkage
orders = datasets["olist_orders_dataset"]

# For one duplicated review_id, check the two orders
sample_rid = dup_check.index[0]
sample_orders = reviews[reviews["review_id"] == sample_rid]["order_id"].tolist()

orders[orders["order_id"].isin(sample_orders)][
    ["order_id", "customer_id", "order_purchase_timestamp", "order_status"]
]

,order_id,customer_id,order_purchase_timestamp,order_status
28241,04a28263e085d399c97ae49e0b477efa,fef2e5e63da9f3e1dd89e8e319468657,2018-02-02 18:01:10,delivered
74048,dfcdfc43867d1c1381bfaf62d6b9c195,a7026133ddbd2e86c83ecd4dfa4dbe01,2018-02-02 18:01:08,delivered


In [ ]:
# Check if any order has multiple rows with same review_id
order_rid_check = (
    reviews
    .groupby(["order_id", "review_id"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
)
print(" (order_id, review_id) :", len(order_rid_check))

 (order_id, review_id) : 0


In [ ]:
# Show a real example with all details
sample_rid = dup_check.index[0]
reviews[reviews["review_id"] == sample_rid][
    ["review_id", "order_id", "review_score", "review_comment_message",
     "review_creation_date", "review_answer_timestamp"]
]

,review_id,order_id,review_score,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23


> Finding:
The review dataset contains multiple review records for some orders,
and review_id is not unique. Therefore, review_id is not treated as
a primary key. The raw review records are retained for analysis.

In [ ]:
# --- Step 0: Create a surrogate key for internal use ---
# A reliable unique identifier for each row, independent of review_id
reviews = reviews.reset_index(drop=True)
reviews["review_record_id"] = reviews.index + 1

# --- Step 1: Flag shared review_ids ---
# Count how many distinct orders each review_id is attached to
rid_order_count = reviews.groupby("review_id")["order_id"].nunique()

# Mark reviews whose review_id appears in more than one order
reviews["is_shared_review_id"] = (
    reviews["review_id"].map(rid_order_count) > 1
)

# --- Step 2: Flag orders with multiple review records ---
# Count how many review records exist per order
order_review_count = reviews.groupby("order_id")["review_id"].count()

# Mark reviews that belong to an order with more than one review record
reviews["order_has_multiple_reviews"] = (
    reviews["order_id"].map(order_review_count) > 1
)

# --- Step 3: Combine into a single quality flag ---
# Priority: shared_review_id > multiple_reviews_per_order > ok
reviews["data_quality_flag"] = "ok"
reviews.loc[
    reviews["order_has_multiple_reviews"],
    "data_quality_flag"
] = "multiple_reviews_per_order"
reviews.loc[
    reviews["is_shared_review_id"],
    "data_quality_flag"
] = "shared_review_id"

# --- Step 4: Quick audit ---
print("Total rows:", len(reviews))
print("\nQuality flag distribution:")
print(reviews["data_quality_flag"].value_counts())
print("\nShared review_id rows:", reviews["is_shared_review_id"].sum())
print("Orders with multiple reviews:", reviews["order_has_multiple_reviews"].sum())

Total rows: 99224

Quality flag distribution:
data_quality_flag
ok                            97172
shared_review_id               1603
multiple_reviews_per_order      449
Name: count, dtype: int64

Shared review_id rows: 1603
Orders with multiple reviews: 1098


In [23]:
# Convert order date columns
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]
orders = datasets["olist_orders_dataset"].copy()
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

# Missing delivery date by status
orders["delivery_missing"] = orders["order_delivered_customer_date"].isna()
pd.crosstab(orders["order_status"], orders["delivery_missing"])

delivery_missing,False,True
order_status,,
approved,0,2
canceled,6,619
created,0,5
delivered,96470,8
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,0,609


In [25]:
# Latest review per order (main rating for the analysis)
latest_review = (
    reviews.sort_values(["order_id", "review_answer_timestamp", "review_id"])
           .groupby("order_id", as_index=False)
           .last()
)
latest_review.shape   # expect 98673 rows

(98673, 7)

## Review Data Quality Investigation — Final Summary

### Dataset Grain

The reviews table is defined at the following grain:

- One row = one review record in the dataset

This means an order_id can appear in multiple rows, and a review_id can also appear in multiple rows.

### Initial Audit Findings

The reviews dataset contains 99,224 records with 98,410 unique `review_id` values, which means `review_id` is not unique in this dataset.

Two distinct quality issues were identified:

1. **Shared review_id across orders:** 789 review_id values are shared across more than one order — 764 of them appear in exactly 2 orders, and 25 appear in exactly 3 orders (764×2 + 25×3 = 1,603 rows, matching the flagged row count). For every one of these groups (verified across all groups, not just a sample):
   - `review_score` is identical
   - `review_comment_message` is identical
   - `review_creation_date` is identical
   - `review_answer_timestamp` is identical
   - `customer_id` differs across every linked order

2. **Multiple reviews per order:** 547 orders contain more than one review record (1,098 records total). These records differ in score, message, or timing, and represent either legitimate multiple reviews or user edits.

### Interpretation

The shared review_id pattern is highly systematic and cannot be explained by user behavior (a user cannot post the identical review, with an identical timestamp, for two different orders under two different customer_ids). The exact root cause cannot be confirmed from the dataset alone. The pattern strongly suggests a data-generation or data-integration issue, but this is an inference, not a verified fact.

For orders with multiple reviews, the reasons vary. Some look like duplicates, some like edits, and some like genuinely independent reviews submitted at different times.

### Decision: Flag, Don't Delete

No rows are deleted at this stage. Instead, three quality flags are added to the reviews table:

| Column | Meaning |
|---|---|
| review_record_id | A surrogate key for internal use (1 to N) |
| is_shared_review_id | True if the same review_id appears in more than one order |
| order_has_multiple_reviews | True if the order has more than one review record |
| data_quality_flag | Summary: `ok`, `shared_review_id`, or `multiple_reviews_per_order` |

### Why Flag Instead of Delete

- At least one of the linked records sharing a review_id is likely valid, but we cannot determine which one.
- Deleting both would lose valid customer experiences.
- Deleting one arbitrarily could bias the analysis.
- Keeping all rows and filtering per analysis is more transparent and reproducible.

### How This Affects Analysis

| Analysis Type | Filtering Strategy |
|---|---|
| Per-order rating analysis | Latest review per order (by `review_answer_timestamp`); sensitivity check against `data_quality_flag == "ok"` only (excludes ~2% of rows) |
| Overall sentiment analysis | Keep all rows, flag-aware |
| Seller-level rating analysis | Aggregate to latest review per order first, then join |
| Customer behavior analysis | Keep all rows, flag-aware |
| Machine learning training | Exclude shared-review rows to avoid label noise |

### Assumptions and Limitations

- We do not know which of the two (or three) linked orders is the "true" owner of a shared review.
- The root cause of the shared review_id pattern is unknown; it may originate from Olist's data collection pipeline.
- `review_id` alone is not a reliable primary key. A surrogate key (`review_record_id`) was created for internal analysis.
- `review_id` + `order_id` is technically unique in this snapshot, but this does not mean it represents the original system's primary key.
- An order with multiple items belonging to different product categories or sellers has its single order-level rating attributed to all of them equally.
- `olist_geolocation_dataset` has 26% duplicate rows and a non-unique `geolocation_zip_code_prefix`; any join to it must first aggregate to one row per zip prefix (e.g. median lat/lng), or downstream row counts will inflate.
- `customer_id` is not a person-level identifier — it is generated per order. Any customer-level analysis (e.g. repeat purchases) must use `customer_unique_id`.

### Next Step

With review-level data quality documented and flagged, the project moves to the analytical layer. The next step is to define the analytical grain for each business question before performing any joins.

This decision reflects a core principle in analytical work:

> Data cleaning is not the same as deleting unusual rows.

| Question | Grain | Tables | Join Key | Pre-aggregation | Metric |
| --- | --- | --- | --- | --- | --- |
| Late deliveries vs ratings | 1 row = 1 order | orders + reviews | order_id | reviews → mean per order | avg rating, low-rating rate |
| Delay severity vs ratings | 1 row = 1 order | orders + reviews | order_id | reviews → mean per order | avg rating by delay bin |
| State-level late/rating | 1 row = 1 order | customers + orders + reviews | customer_id + order_id | reviews → mean per order | rate by state |
| Price/freight vs ratings | 1 row = 1 order | order_items + orders + reviews | order_id | order_items → sum per order; reviews → mean per order | avg rating by price bin |
| Category vs ratings | 1 row = 1 order-category | order_items + products + reviews | order_id + product_id | reviews → mean per order; then join | low-rating rate by category |
| Installments vs ratings | 1 row = 1 order | payments + orders + reviews | order_id | payments → max installments per order; reviews → mean | avg rating by installment bin |
| Seller low-rating rate | 1 row = 1 seller | sellers + order_items + reviews | seller_id + order_id | reviews → mean per order; then aggregate per seller | low-rating rate per seller |
| Repeat customers vs ratings | 1 row = 1 customer | customers + orders + reviews | customer_unique_id | orders → count per customer; reviews → mean per customer | avg rating: repeat vs one-time |